# 📓 Notebook 19 — LangChain RAG---**Phase 7 · Industry Frameworks**> LangChain provides a complete RAG pipeline out of the box — document loaders, text splitters, embeddings, vector stores, retrievers, and chains. This is how production RAG is built in industry.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Document loaders | Load PDF, TXT, CSV, Web, and more || Text splitters | Production chunking strategies || Embeddings | LangChain embedding wrappers || Vector stores | ChromaDB, FAISS via LangChain || Retrieval chains | LCEL-based RAG pipelines || Conversational RAG | RAG with follow-up questions |### 🏗️ Build: Production RAG Pipeline with LangChain

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode, get_langchain_llm, get_langchain_embeddings

# LangChain LLM — auto-routes through proxy when available
llm = get_langchain_llm()

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")
print(f"   LangChain LLM: {llm.model_name}")

---## 2. Document LoadersLangChain provides 100+ loaders for different data sources.

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader, WebBaseLoaderfrom langchain_core.documents import Document# Create sample documents for demoimport tempfile, osDEMO_DIR = tempfile.mkdtemp()# Create sample fileswith open(os.path.join(DEMO_DIR, "ai_basics.txt"), "w") as f:    f.write("""Artificial Intelligence (AI) FundamentalsMachine Learning is a subset of AI that enables computers to learn from data without explicit programming.Deep Learning uses neural networks with multiple layers to learn hierarchical representations.Key architectures include CNNs for vision, RNNs for sequences, and Transformers for everything.The Transformer architecture, introduced in 2017's "Attention Is All You Need" paper,revolutionized NLP. It uses self-attention mechanisms to process all tokens in parallel,unlike RNNs which process sequentially.Large Language Models (LLMs) are transformer-based models trained on billions of tokens.GPT, BERT, T5, and LLaMA are popular LLM families. They can be fine-tuned or usedwith in-context learning (few-shot prompting) for downstream tasks.Retrieval Augmented Generation (RAG) grounds LLM responses in external knowledge,reducing hallucination and providing up-to-date information. RAG combines a retriever(finds relevant documents) with a generator (produces the answer).""")with open(os.path.join(DEMO_DIR, "python_guide.txt"), "w") as f:    f.write("""Python Programming GuidePython is a high-level, interpreted programming language created by Guido van Rossum in 1991.It emphasizes code readability and supports multiple programming paradigms.Key Python Features:- Dynamic typing with optional type hints (PEP 484)- List comprehensions and generator expressions- Decorators for cross-cutting concerns- Context managers (with statement) for resource management- async/await for asynchronous programmingPopular Python Libraries for AI:- NumPy: Numerical computing with N-dimensional arrays- Pandas: Data manipulation and analysis- Scikit-learn: Classical machine learning algorithms- PyTorch: Deep learning framework (Facebook/Meta)- TensorFlow: Deep learning framework (Google)- LangChain: Framework for LLM application development- FastAPI: Modern web framework for building APIs""")# Load documentsloader1 = TextLoader(os.path.join(DEMO_DIR, "ai_basics.txt"))loader2 = TextLoader(os.path.join(DEMO_DIR, "python_guide.txt"))docs = loader1.load() + loader2.load()print(f"📄 Loaded {len(docs)} documents")for d in docs:    print(f"   • {os.path.basename(d.metadata['source'])}: {len(d.page_content)} chars")

---## 3. Text Splitters — Production Chunking| Splitter | Strategy | Best For ||----------|----------|----------|| `RecursiveCharacterTextSplitter` | Splits by `\n\n` → `\n` → ` ` | General-purpose (default choice) || `CharacterTextSplitter` | Splits by single separator | Simple documents || `TokenTextSplitter` | Splits by token count | Token-budget-aware systems || `MarkdownHeaderTextSplitter` | Splits by headers | Markdown/docs || `SemanticChunker` | Splits by semantic similarity | Meaning-preserving chunks |

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter# RecursiveCharacterTextSplitter — the production defaultrecursive_splitter = RecursiveCharacterTextSplitter(    chunk_size=300,    chunk_overlap=50,    separators=["\n\n", "\n", ". ", " ", ""],    length_function=len,)chunks = recursive_splitter.split_documents(docs)print(f"📦 Split {len(docs)} documents into {len(chunks)} chunks")print(f"   Chunk sizes: {[len(c.page_content) for c in chunks]}")# Show first few chunksfor i, chunk in enumerate(chunks[:3]):    print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")    print(f"   Source: {os.path.basename(chunk.metadata['source'])}")    print(f"   Text: {chunk.page_content[:100]}...")

---## 4. Vector Stores — Embed and Store

In [ ]:
from langchain_chroma import Chromaimport tempfile# Create ChromaDB vector store from chunkspersist_dir = os.path.join(tempfile.gettempdir(), "langchain_rag_demo")vectorstore = Chroma.from_documents(    documents=chunks,    embedding=embeddings,    persist_directory=persist_dir,    collection_name="course_demo",)print(f"✅ Vector store created: {vectorstore._collection.count()} vectors")

In [ ]:
# Similarity searchquery = "What is RAG?"results = vectorstore.similarity_search_with_score(query, k=3)print(f"🔍 Query: '{query}'")print(f"   Found {len(results)} results:\n")for doc, score in results:    print(f"   [{score:.4f}] {doc.page_content[:100]}...")    print(f"           Source: {os.path.basename(doc.metadata.get('source', 'unknown'))}")    print()

---## 5. Retriever + Chain = RAG Pipeline

In [ ]:
from langchain_core.prompts import ChatPromptTemplatefrom langchain_core.output_parsers import StrOutputParserfrom langchain_core.runnables import RunnablePassthrough# Create retrieverretriever = vectorstore.as_retriever(    search_type="similarity",    search_kwargs={"k": 3})# RAG promptrag_prompt = ChatPromptTemplate.from_messages([    ("system", """You are a knowledgeable AI assistant. Answer the question based ONLY on the provided context.If the context doesn't contain the answer, say "I don't have enough information to answer that."Context:{context}"""),    ("human", "{question}"),])# Format retrieved docs into context stringdef format_docs(docs):    return "\n\n".join([        f"[Source: {os.path.basename(d.metadata.get('source', 'unknown'))}]\n{d.page_content}"         for d in docs    ])# LCEL RAG chainrag_chain = (    {"context": retriever | format_docs, "question": RunnablePassthrough()}    | rag_prompt    | llm    | StrOutputParser())# Test the RAG pipelinequestions = [    "What is RAG and why is it used?",    "What are the key Python libraries for AI?",    "How do Transformers work?",    "What is the capital of France?",  # Not in our documents]print("🤖 LangChain RAG Pipeline")print("=" * 60)for q in questions:    print(f"\n❓ {q}")    answer = rag_chain.invoke(q)    print(f"📝 {answer}")

---## 6. Conversational RAG (with Memory)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholderfrom langchain_core.chat_history import InMemoryChatMessageHistoryfrom langchain_core.runnables.history import RunnableWithMessageHistory# Contextualize question using chat historycontextualize_prompt = ChatPromptTemplate.from_messages([    ("system", "Given the chat history and latest question, reformulate the question to be standalone. Don't answer it, just reformulate it if needed."),    MessagesPlaceholder(variable_name="chat_history"),    ("human", "{input}"),])# Standalone question chainhistory_aware_retriever_chain = contextualize_prompt | llm | StrOutputParser()# Full conversational RAG chainconv_rag_prompt = ChatPromptTemplate.from_messages([    ("system", "Answer based on the context. If unsure, say so.\n\nContext:\n{context}"),    MessagesPlaceholder(variable_name="chat_history"),    ("human", "{input}"),])def get_context(input_dict):    # If there's chat history, reformulate the question    if input_dict.get("chat_history"):        standalone = history_aware_retriever_chain.invoke(input_dict)    else:        standalone = input_dict["input"]    docs = retriever.invoke(standalone)    return format_docs(docs)from langchain_core.runnables import RunnableLambdaconv_rag_chain = (    RunnablePassthrough.assign(context=RunnableLambda(get_context))    | conv_rag_prompt    | llm    | StrOutputParser())# Wrap with chat historystore = {}def get_history(session_id):    if session_id not in store:        store[session_id] = InMemoryChatMessageHistory()    return store[session_id]conversational_chain = RunnableWithMessageHistory(    conv_rag_chain,    get_history,    input_messages_key="input",    history_messages_key="chat_history",)# Test conversational RAGconfig = {"configurable": {"session_id": "demo_session"}}conversation = [    "What is RAG?",    "How does it reduce hallucination?",              # Follow-up    "What Python libraries can I use to build it?",   # Follow-up]print("💬 Conversational RAG Demo")print("=" * 60)for q in conversation:    print(f"\n👤 {q}")    answer = conversational_chain.invoke({"input": q}, config=config)    print(f"🤖 {answer}")

---## 📝 Interview Quiz — LangChain RAG### Q1: Walk me through a LangChain RAG pipeline step by step.<details><summary>💡 Answer</summary>1. **Load** — `DocumentLoader` loads raw files (PDF, TXT, Web)2. **Split** — `TextSplitter` chunks documents (RecursiveCharacter is default)3. **Embed** — `Embeddings` model converts chunks to vectors4. **Store** — `VectorStore` (Chroma, FAISS, Pinecone) stores vectors5. **Retrieve** — `Retriever` finds top-k similar chunks for a query6. **Generate** — `ChatModel` answers using retrieved contextIn LCEL:```pythonchain = (    {"context": retriever | format_docs, "question": RunnablePassthrough()}    | prompt | llm | StrOutputParser())```</details>### Q2: How do you handle follow-up questions in RAG?<details><summary>💡 Answer</summary>**Problem:** "What libraries support it?" — "it" refers to something from earlier in the conversation.**Solution — History-aware retriever:**1. Take chat history + new question2. Use LLM to reformulate into a standalone question: "What libraries support RAG?"3. Use the standalone question for retrieval4. Generate answer with both retrieved context and chat historyThis is the `create_history_aware_retriever` pattern in LangChain.</details>### Q3: `RecursiveCharacterTextSplitter` vs `SemanticChunker` — compare.<details><summary>💡 Answer</summary>| Aspect | RecursiveCharacter | SemanticChunker ||--------|--------------------|-----------------|| How | Splits by separators (`\n\n`, `\n`, `. `) | Splits where embedding similarity drops || Speed | Fast (string ops only) | Slow (needs embeddings) || Quality | Good for structured text | Better for unstructured text || Cost | Free | Embedding API calls || Default choice | Yes (most projects) | When precision matters |**Recommendation:** Start with RecursiveCharacter. Switch to Semantic only if retrieval quality is a problem.</details>### Q4: How do you evaluate a LangChain RAG pipeline?<details><summary>💡 Answer</summary>Use **RAGAS** or **LangSmith evaluations**:1. **Context Recall** — Did we retrieve the right chunks?2. **Context Precision** — Are retrieved chunks relevant (not noise)?3. **Faithfulness** — Is the answer supported by the context?4. **Answer Relevance** — Does the answer address the question?LangSmith integration:```pythonfrom langsmith import Clientclient = Client()# Create test dataset → Run chain on all examples → View results in LangSmith dashboard```</details>

---## ✅ Notebook 19 Summary| Concept | Key Takeaway ||---------|-------------|| Document loaders | 100+ loaders; TextLoader, PyPDFLoader, WebBaseLoader || Text splitters | RecursiveCharacter is the production default || Vector stores | Chroma, FAISS via LangChain wrappers || RAG chain | `retriever \| format_docs` → `prompt \| llm \| parser` || Conversational RAG | Reformulate follow-ups into standalone questions || Evaluation | RAGAS metrics + LangSmith for production |### ➡️ Next: [Notebook 20 — LangGraph](./20_langgraph.ipynb)